# Dense Retrieval: Semantic Search with Embeddings

In the first retrieval notebook, we ranked text using explicit word overlap. That made the mechanics easy to inspect, but it struggled when a query and a useful passage expressed the same idea with different words. This notebook introduces **dense retrieval**: represent queries and chunks as numeric vectors, then rank chunks by vector similarity.

We will first build a tiny, hand-designed semantic encoder so every vector is explainable and the notebook runs with the Python standard library. This encoder is a teaching device, **not** a learned model. Then we will connect the same retrieval pipeline to a pretrained sentence-embedding model as an optional hands-on lab.

## Learning goals

By the end, you should be able to:

- Explain how an encoder maps text to a fixed-length vector and why query and chunk vectors must share the same embedding space.
- Calculate cosine similarity and use it to rank candidate chunks.
- Compare dense retrieval with the lexical-overlap baseline from Notebook 01 on paraphrased queries.
- Explain why a similarity score is not a probability, relevance judgment, or guarantee of factual support.
- Identify important model, normalization, chunking, and evaluation choices in a dense retrieval workflow.
- Run an optional pretrained-model example using Sentence Transformers when the package and model are available.

## 1. Connect this lesson to retrieval fundamentals

The retrieval pipeline is still **query → candidate chunks → scores → ranking → top-K results**. Dense retrieval changes how text is represented and scored; it does not change the job retrieval is trying to do.

Notebook 01 used a transparent lexical score: count query terms that also occur in a chunk. That method is sensitive to wording. Dense retrieval uses an encoder trained on language examples to place texts with related meanings near one another in a vector space. The encoder supplies the representation; a similarity function compares vectors; the retriever sorts the resulting scores.

| Component | Role in dense retrieval |
| --- | --- |
| Text encoder | Converts text into a fixed-length numeric vector. |
| Chunk embedding | Vector representation stored for each corpus chunk. |
| Query embedding | Vector representation computed for the user's query. |
| Similarity function | Compares query and chunk vectors. |
| Ranking and top-K | Orders candidate chunks and selects the first K. |

The encoder is not the retriever by itself. A useful embedding model, compatible query/document encoding, a good corpus, and sensible retrieval settings all matter.

### Sparse versus dense representations

A sparse representation typically has a very large vocabulary-sized space, with most coordinates zero; lexical methods such as bag-of-words and BM25 use this kind of term-oriented evidence. A dense embedding has a comparatively compact, fixed number of learned numeric dimensions, and most coordinates may be nonzero. Its dimensions usually do not have simple human-readable meanings such as “password” or “refund.”

Dense retrieval can match related wording without exact token overlap, but it can also blur distinctions, miss rare exact identifiers, or rank text that is topically similar but does not answer the question. It complements lexical search; it does not make careful retrieval design unnecessary.

## 2. Reuse the teaching corpus

We reuse the same ten chunks and stable IDs from Notebook 01 so comparisons are meaningful. In a real project, these chunk texts would usually be produced by a document-ingestion and chunking pipeline, then stored with IDs and metadata in a vector index or database.

In [1]:
corpus = [
    {'chunk_id': 'C01', 'source': 'account_help', 'topic': 'password',
     'text': 'To reset your password, choose Forgot password on the sign-in page. We send a reset link to your registered email address.'},
    {'chunk_id': 'C02', 'source': 'account_help', 'topic': 'security',
     'text': 'For account security, turn on two-factor authentication and never share your verification code with anyone.'},
    {'chunk_id': 'C03', 'source': 'shopping_policy', 'topic': 'refunds',
     'text': 'Refunds for eligible orders are returned to the original payment method after the returned item is inspected.'},
    {'chunk_id': 'C04', 'source': 'shipping_guide', 'topic': 'delivery',
     'text': 'Standard shipping usually takes five to seven business days. Express shipping is available at checkout.'},
    {'chunk_id': 'C05', 'source': 'shopping_policy', 'topic': 'returns',
     'text': 'Unopened products can be returned within thirty days of delivery. Include the order number in your return request.'},
    {'chunk_id': 'C06', 'source': 'tenant_handbook', 'topic': 'lease',
     'text': 'The security deposit is due when the lease is signed. The deposit amount is listed in the signed lease agreement.'},
    {'chunk_id': 'C07', 'source': 'cycling_guide', 'topic': 'safety',
     'text': 'Choose a bicycle helmet that fits snugly and sits level on the head. Replace a helmet after a serious impact.'},
    {'chunk_id': 'C08', 'source': 'account_help', 'topic': 'support',
     'text': 'Contact support if you cannot access the email address linked to your account.'},
    {'chunk_id': 'C09', 'source': 'food_guide', 'topic': 'allergies',
     'text': 'Read the ingredient label carefully if you have a food allergy. Contact the manufacturer when allergen information is unclear.'},
    {'chunk_id': 'C10', 'source': 'account_help', 'topic': 'limitations',
     'text': 'Archived note: it describes how password reset worked in an old unrelated system, but gives no current account recovery instructions.'},
]

print(f'{len(corpus)} chunks ready for embedding.')
print(corpus[0]['chunk_id'], corpus[0]['text'])

10 chunks ready for embedding.
C01 To reset your password, choose Forgot password on the sign-in page. We send a reset link to your registered email address.


### Query and relevance judgments

The query below deliberately avoids the words “password” and “reset.” In the labels, `C01` is the useful procedure and `C10` is a related but insufficient archived note. Relevance is judged against the user's need, not similarity alone. We will use the same judgment when we compare retrieval methods.

In [2]:
query = 'How can I access my account after forgetting the login secret?'
relevant_chunk_ids = {'C01'}

print('Query:', query)
print('Relevant chunk IDs:', sorted(relevant_chunk_ids))

Query: How can I access my account after forgetting the login secret?
Relevant chunk IDs: ['C01']


## 3. What does an embedding represent?

An embedding is a vector: an ordered list of numbers such as `[0.12, -0.08, ...]`. A model produces a vector of a fixed dimension for each input. During training, the model learns a geometry in which certain text relationships are reflected by vector relationships. Similarity is an imperfect proxy for semantic relatedness, not an explicit proof that one text answers another.

For retrieval, we need vectors for both query and corpus chunks. They must be produced in a **compatible embedding space**—usually by the same model and compatible encoding instructions. Comparing a query vector from one unrelated model to chunk vectors from another has no useful interpretation, even if the dimensions happen to match.

To make the geometry visible without downloads or libraries, the next cells define a miniature encoder with nine hand-authored concept dimensions. It assigns related words to the same concept. Real models learn rich contextual patterns from large training data; this miniature encoder only follows our explicit mapping.

### A hand-designed concept space (teaching device only)

Each mapped token activates one of nine concepts. Words such as `password`, `login`, and `secret` activate the same credentials dimension; `reset`, `forgot`, and `regain` activate the recovery dimension. We average the activated vectors and then normalize. This makes a simple paraphrase searchable while keeping the encoding process auditable.

**Important limitation:** this is not a trained semantic model, not a general-purpose language understanding system, and not a recommended production encoder. It is equivalent to us manually writing down a tiny synonym dictionary. It cannot infer relationships we did not encode.

In [3]:
import math
import re

DIMENSION_NAMES = [
    'credentials', 'account_recovery', 'email', 'delivery', 'refund',
    'return', 'lease_payment', 'bicycle_safety', 'food_allergy',
]
DIMENSION_INDEX = {name: index for index, name in enumerate(DIMENSION_NAMES)}

TOKEN_TO_CONCEPT = {
    'password': 'credentials', 'login': 'credentials', 'secret': 'credentials',
    'credential': 'credentials', 'credentials': 'credentials',
    'reset': 'account_recovery', 'forgot': 'account_recovery',
    'forgetting': 'account_recovery', 'regain': 'account_recovery',
    'recover': 'account_recovery', 'recovery': 'account_recovery',
    'email': 'email', 'link': 'email', 'shipping': 'delivery',
    'delivery': 'delivery', 'refund': 'refund', 'refunds': 'refund',
    'return': 'return', 'returned': 'return', 'lease': 'lease_payment',
    'deposit': 'lease_payment', 'helmet': 'bicycle_safety',
    'bicycle': 'bicycle_safety', 'allergy': 'food_allergy',
    'allergen': 'food_allergy',
}

def tokenize(text):
    return re.findall(r'[a-z0-9]+', text.lower())

def l2_normalize(vector):
    norm = math.sqrt(sum(value * value for value in vector))
    if norm == 0:
        return [0.0] * len(vector)
    return [value / norm for value in vector]

def teaching_encode(text):
    vector = [0.0] * len(DIMENSION_NAMES)
    mapped_count = 0
    for token in tokenize(text):
        concept = TOKEN_TO_CONCEPT.get(token)
        if concept is not None:
            vector[DIMENSION_INDEX[concept]] += 1.0
            mapped_count += 1
    if mapped_count:
        vector = [value / mapped_count for value in vector]
    return l2_normalize(vector)

print('Embedding dimension:', len(teaching_encode(query)))
print('Query embedding:', [round(value, 3) for value in teaching_encode(query)])

Embedding dimension: 9
Query embedding: [0.894, 0.447, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


The vector has one coordinate per concept. Unknown words contribute nothing in this toy encoder. In learned models, tokenization and contextual representation are substantially more sophisticated; a model may use subword tokens and determine a token's representation from surrounding words.

### Inspect the chunk vectors

We encode each chunk once and keep its vector next to the chunk record. This mirrors a common indexing workflow: embed documents during ingestion, store the vectors, then embed each new query at search time. If the corpus or embedding model changes, the stored vectors may need to be regenerated.

In [4]:
chunk_vectors = {
    chunk['chunk_id']: teaching_encode(chunk['text'])
    for chunk in corpus
}

for chunk_id in ('C01', 'C10', 'C05'):
    vector = chunk_vectors[chunk_id]
    active = {
        DIMENSION_NAMES[index]: round(value, 3)
        for index, value in enumerate(vector)
        if value != 0
    }
    print(chunk_id, active)

C01 {'credentials': 0.485, 'account_recovery': 0.728, 'email': 0.485}
C10 {'credentials': 0.447, 'account_recovery': 0.894}
C05 {'delivery': 0.447, 'return': 0.894}


## 4. Cosine similarity

Cosine similarity compares the angle between two vectors:

`cosine(a, b) = (a · b) / (||a|| × ||b||)`

The dot product adds coordinate-wise products; the norms scale by vector length. For nonzero vectors, cosine lies between -1 and 1 mathematically. Some embedding models and data produce mostly nonnegative similarities in practice, but that is not a universal guarantee. A larger cosine score means a smaller angle and, under this scoring rule, a closer direction.

When both vectors have already been L2-normalized to unit length, their dot product equals cosine similarity. We will implement both pieces explicitly to show what is being computed.

In [5]:
def dot_product(left, right):
    if len(left) != len(right):
        raise ValueError('Vectors must have the same dimension')
    return sum(a * b for a, b in zip(left, right))

def cosine_similarity(left, right):
    if len(left) != len(right):
        raise ValueError('Vectors must have the same dimension')
    left_norm = math.sqrt(dot_product(left, left))
    right_norm = math.sqrt(dot_product(right, right))
    if left_norm == 0 or right_norm == 0:
        return 0.0
    return dot_product(left, right) / (left_norm * right_norm)

example_a = [1.0, 0.0, 0.0]
example_b = [1.0, 1.0, 0.0]
example_c = [0.0, 1.0, 0.0]

print('same direction:', round(cosine_similarity(example_a, example_a), 3))
print('partly aligned:', round(cosine_similarity(example_a, example_b), 3))
print('orthogonal:', round(cosine_similarity(example_a, example_c), 3))
assert abs(cosine_similarity(example_a, example_a) - 1.0) < 1e-12
assert abs(cosine_similarity(example_a, example_c)) < 1e-12

same direction: 1.0
partly aligned: 0.707
orthogonal: 0.0


For this teaching encoder, negative coordinates are not produced, but real embedding models can produce negative values. The zero-vector guard is included because a text with no recognized terms has no direction. Returning zero in that edge case is an explicit convention for this notebook, not a mathematical cosine value (cosine is undefined for a zero vector). A production pipeline should detect and investigate empty or invalid embeddings rather than treating them as good matches.

### A geometric intuition

Imagine each vector as an arrow from the origin. If query and chunk arrows point in similar directions, cosine similarity is high. Their lengths do not affect cosine after normalization; the angle does. If two vectors point along unrelated axes, the cosine is near zero. This geometric picture is useful, but the semantic meaning of the axes is learned by a real model and cannot usually be inspected as a simple list of concepts.

## 5. Embed the query, score chunks, and rank

For each search, we encode the query with the same encoder used for the chunks, calculate cosine similarity against every stored chunk vector, and sort from highest to lowest. As in Notebook 01, ties are made deterministic by sorting equal scores by `chunk_id` ascending. The teaching example scans every chunk; vector indexes can accelerate candidate search over large collections.

In [6]:
def dense_rank(query_text, chunks, vectors, encoder=teaching_encode):
    query_vector = encoder(query_text)
    ranked = []
    for chunk in chunks:
        chunk_vector = vectors[chunk['chunk_id']]
        ranked.append({
            **chunk,
            'score': cosine_similarity(query_vector, chunk_vector),
        })
    return sorted(ranked, key=lambda item: (-item['score'], item['chunk_id']))

def dense_retrieve(query_text, chunks, vectors, k=3, encoder=teaching_encode):
    if k < 0:
        raise ValueError('k must be non-negative')
    return dense_rank(query_text, chunks, vectors, encoder=encoder)[:k]

dense_results = dense_retrieve(query, corpus, chunk_vectors, k=5)
for rank, item in enumerate(dense_results, start=1):
    print(f"{rank}. {item['chunk_id']} cosine={item['score']:.3f} | {item['text']}")

1. C10 cosine=0.800 | Archived note: it describes how password reset worked in an old unrelated system, but gives no current account recovery instructions.
2. C01 cosine=0.759 | To reset your password, choose Forgot password on the sign-in page. We send a reset link to your registered email address.
3. C02 cosine=0.000 | For account security, turn on two-factor authentication and never share your verification code with anyone.
4. C03 cosine=0.000 | Refunds for eligible orders are returned to the original payment method after the returned item is inspected.
5. C04 cosine=0.000 | Standard shipping usually takes five to seven business days. Express shipping is available at checkout.


The query uses “login secret” and “forgetting”; the relevant chunk says “password” and “reset.” Our explicit mapping places those expressions in related dimensions, so the query can find `C01` without exact word overlap. The query also contains exact terms such as “access” and “account” that occur in other chunks, so lexical search has plausible but incomplete matches. `C10` also discusses password recovery and can score highly despite being insufficient evidence. Better semantic matching helps with vocabulary mismatch; it does not perfectly distinguish intent, negation, time, or whether a passage actually supports an answer.

### Top-K and ranked scores

Top-K selection works just as it did in the first notebook. K controls how many ranked candidates we return, not how many are relevant. A larger K can increase recall, but it can also add weaker matches. Dense scores should be interpreted in the context of the model, metric, and evaluation data; do not assume a universal cutoff such as 0.7 works for every model or query.

In [7]:
for k in (1, 3, 6):
    results = dense_retrieve(query, corpus, chunk_vectors, k=k)
    returned_ids = [item['chunk_id'] for item in results]
    print(f'K={k}: {returned_ids}')

assert dense_retrieve(query, corpus, chunk_vectors, k=0) == []
assert len(dense_retrieve(query, corpus, chunk_vectors, k=100)) == len(corpus)

K=1: ['C10']
K=3: ['C10', 'C01', 'C02']
K=6: ['C10', 'C01', 'C02', 'C03', 'C04', 'C05']


## 6. Compare dense retrieval with lexical overlap

Notebook 01's overlap score counts exact token matches. We'll repeat that simple baseline here to compare rankings on identical chunks. This is a teaching comparison, not a benchmark between a trained embedding model and BM25.

In [8]:
def lexical_score(query_text, chunk_text):
    return len(set(tokenize(query_text)) & set(tokenize(chunk_text)))

def lexical_rank(query_text, chunks):
    ranked = []
    for chunk in chunks:
        ranked.append({**chunk, 'score': lexical_score(query_text, chunk['text'])})
    return sorted(ranked, key=lambda item: (-item['score'], item['chunk_id']))

lexical_results = lexical_rank(query, corpus)
dense_results = dense_rank(query, corpus, chunk_vectors)

print('Lexical top 5:')
for item in lexical_results[:5]:
    print(f"  {item['chunk_id']} overlap={item['score']} | {item['text']}")

print('\nDense top 5:')
for item in dense_results[:5]:
    print(f"  {item['chunk_id']} cosine={item['score']:.3f} | {item['text']}")

Lexical top 5:
  C08 overlap=3 | Contact support if you cannot access the email address linked to your account.
  C03 overlap=2 | Refunds for eligible orders are returned to the original payment method after the returned item is inspected.
  C05 overlap=2 | Unopened products can be returned within thirty days of delivery. Include the order number in your return request.
  C07 overlap=2 | Choose a bicycle helmet that fits snugly and sits level on the head. Replace a helmet after a serious impact.
  C10 overlap=2 | Archived note: it describes how password reset worked in an old unrelated system, but gives no current account recovery instructions.

Dense top 5:
  C10 cosine=0.800 | Archived note: it describes how password reset worked in an old unrelated system, but gives no current account recovery instructions.
  C01 cosine=0.759 | To reset your password, choose Forgot password on the sign-in page. We send a reset link to your registered email address.
  C02 cosine=0.000 | For account s

The paraphrase has little or no exact overlap with the relevant chunk, so lexical search may fail to place `C01` near the top. The teaching encoder can match its manually listed synonyms. Notice that other chunks may also receive zero lexical score or small dense scores; a score is useful for ordering candidates, but neither method knows the user's full intent unless that intent is represented in its input and training.

### Compare using the same relevance label

We will reuse the precision@K and recall@K definitions from Notebook 01. With only one relevant chunk for this query, recall@K is 1 as soon as that chunk appears in the top K. Precision@K reflects how much of the returned list is relevant. These metrics allow a fairer comparison than judging scores by visual magnitude.

In [9]:
def precision_at_k(results, relevant_ids, k):
    if k <= 0:
        raise ValueError('k must be positive')
    selected = results[:k]
    if not selected:
        return 0.0
    hits = sum(item['chunk_id'] in relevant_ids for item in selected)
    return hits / len(selected)

def recall_at_k(results, relevant_ids, k):
    if k <= 0:
        raise ValueError('k must be positive')
    if not relevant_ids:
        raise ValueError('relevant_ids must not be empty')
    hits = sum(item['chunk_id'] in relevant_ids for item in results[:k])
    return hits / len(relevant_ids)

for method_name, results in [('lexical', lexical_results), ('dense teaching encoder', dense_results)]:
    print(method_name)
    for k in (1, 3, 5):
        print(f'  P@{k}={precision_at_k(results, relevant_chunk_ids, k):.2f}, '
              f'R@{k}={recall_at_k(results, relevant_chunk_ids, k):.2f}')

assert lexical_results[0]['chunk_id'] != 'C01'
assert dense_results[0]['chunk_id'] != 'C01'
assert 'C01' in [item['chunk_id'] for item in dense_results[:3]]

lexical
  P@1=0.00, R@1=0.00
  P@3=0.00, R@3=0.00
  P@5=0.00, R@5=0.00
dense teaching encoder
  P@1=0.00, R@1=0.00
  P@3=0.33, R@3=1.00
  P@5=0.20, R@5=1.00


Interpret the table rather than treating it as a universal result. It measures one query, one tiny corpus, one manually authored encoder, and one relevance label. A useful model comparison needs a representative set of queries, reviewed relevance judgments, consistent candidate coverage, and the same evaluation protocol for every method.

## 7. Important dense-retrieval design choices

### Query and document encoding

Some models use the same encoding behavior for both sides. Others are trained for **asymmetric retrieval** and expect different query and document instructions, prefixes, or model methods. Follow the model's documented usage. Using the right model but the wrong query/document format can reduce retrieval quality.

### Normalization and similarity metric

Cosine similarity is common. If vectors are normalized, dot-product ranking produces the same ordering as cosine. If vectors are not normalized, dot product also depends on vector magnitude. Euclidean distance, cosine similarity, and dot product have different score directions and interpretations. Match the database/index metric to the embedding model's intended use and representation.

### Embedding model and domain

Model size, training objective, language coverage, domain, query/document length, and instruction-following behavior can all matter. A generally capable model is not automatically best for legal, medical, code, or organization-specific content. Evaluate candidate models on examples that reflect the real search task.

### Chunking and metadata

Embeddings represent the chunk text provided to the model. A chunk that omits a key qualifier or combines unrelated passages can yield a poor vector for the intended query. Metadata filters can narrow the candidate set, but overly restrictive filters can exclude relevant chunks before vector ranking.

## 8. Optional lab: use a pretrained sentence embedding model

The teaching encoder makes the mechanics inspectable, but it does not learn meaning. For a practical dense retriever, install the optional dependency in the active Jupyter kernel, then run the model cell below. Sentence Transformers will download the model the first time it is used; this needs network access and may take a while.

In a notebook cell, install with:

```python
%pip install sentence-transformers
```

This lab uses `sentence-transformers/all-MiniLM-L6-v2` as a compact demonstration model. Check the model card and library documentation for intended usage, licensing, limitations, and any query/document instructions before applying a model to a real system. This choice is for learning, not a claim that it is best for every task.

In [10]:
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    print('Optional model lab not run: install sentence-transformers with %pip, then rerun this cell.')
else:
    pretrained_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    print('Loaded model: sentence-transformers/all-MiniLM-L6-v2')

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded model: sentence-transformers/all-MiniLM-L6-v2


When the model is loaded, embed the corpus in batches. Normalizing embeddings here makes dot products equal cosine similarities. Batch encoding is more efficient than encoding each chunk individually for larger collections. Keep IDs and vectors aligned; in production, persist each vector with its chunk ID and metadata in the vector database.

In [11]:
if 'pretrained_model' in globals():
    pretrained_texts = [chunk['text'] for chunk in corpus]
    pretrained_vectors = pretrained_model.encode(
        pretrained_texts,
        batch_size=16,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    pretrained_vectors_by_id = {
        chunk['chunk_id']: pretrained_vectors[index]
        for index, chunk in enumerate(corpus)
    }
    print('Embedding matrix shape:', pretrained_vectors.shape)
    print('First vector L2 norm:', round(float((pretrained_vectors[0] ** 2).sum() ** 0.5), 5))
else:
    print('Skipped: load the optional pretrained model above first.')

Embedding matrix shape: (10, 384)
First vector L2 norm: 1.0


Encode the query with the same model, then compute exact cosine scores against all chunk vectors. Because both sides are normalized, the dot product gives cosine similarity. This is a small exact scan; a vector database or approximate-nearest-neighbor index becomes useful as the corpus grows.

In [12]:
if 'pretrained_model' in globals():
    import numpy as np

    query_vector = pretrained_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )[0]
    model_scores = pretrained_vectors @ query_vector
    model_order = sorted(
        range(len(corpus)),
        key=lambda index: (-float(model_scores[index]), corpus[index]['chunk_id']),
    )
    for rank, index in enumerate(   model_order[:5], start=1):
        chunk = corpus[index]
        print(f"{rank}. {chunk['chunk_id']} cosine={model_scores[index]:.3f} | {chunk['text']}")
else:
    print('Skipped: load the optional pretrained model above first.')

1. C01 cosine=0.595 | To reset your password, choose Forgot password on the sign-in page. We send a reset link to your registered email address.
2. C08 cosine=0.524 | Contact support if you cannot access the email address linked to your account.
3. C10 cosine=0.487 | Archived note: it describes how password reset worked in an old unrelated system, but gives no current account recovery instructions.
4. C02 cosine=0.375 | For account security, turn on two-factor authentication and never share your verification code with anyone.
5. C06 cosine=0.144 | The security deposit is due when the lease is signed. The deposit amount is listed in the signed lease agreement.


### Compare the learned model with the teaching encoder

Do not expect identical rankings. The hand-designed encoder knows only the synonym groups we wrote; the pretrained model learned a much richer representation, but it may still rank an insufficient passage highly. Inspect the returned text, calculate metrics on multiple labeled queries, and test on your own corpus before drawing conclusions.

If you change the model, corpus preprocessing, query instructions, or normalization behavior, regenerate embeddings consistently. Never compare a query from one model/version to chunks embedded with another unless the model explicitly supports that setup.

## 9. Common dense-retrieval failure modes

| Failure | Why it happens | What to investigate |
| --- | --- | --- |
| Related but non-answer passage ranks highly | Similarity captures topical or semantic closeness, not whether the chunk supports the exact answer. | Improve relevance labels, use reranking, inspect context, or combine with lexical constraints. |
| Exact identifier is missed | Embeddings can blur rare names, codes, versions, or numbers. | Add lexical retrieval or exact-match features. |
| Query and document vectors are incompatible | Different models, versions, prompts, or preprocessing were used. | Re-embed consistently and follow model-specific retrieval guidance. |
| Empty or malformed vector | Input was empty, truncated, invalid, or not encoded as expected. | Validate text, vector dimension, finite values, and vector norm. |
| Relevant evidence is outside top-K | Ranking, chunking, candidate generation, or K is inadequate. | Inspect labeled queries and tune the full retrieval pipeline. |
| Model performs poorly on a domain | Training data or objective does not match the search task. | Evaluate domain-appropriate models and consider task-specific adaptation. |

A similarity score cannot tell the generator that evidence is absent or contradictory. For robust RAG, retrieval quality and answer grounding must be evaluated separately.

## 10. Exercises

1. Change the query to “What is the return window for unopened items?” Compare lexical and dense teaching-encoder rankings. Which terms or concepts explain the result?
2. Add a synonym to `TOKEN_TO_CONCEPT` that helps the bicycle query. Test before and after, and explain why changing this dictionary changes results.
3. Add an irrelevant chunk that mentions both account recovery and a historical password reset. Does similarity alone reliably reject it?
4. Create a query with no mapped concepts. Inspect its zero vector and explain why cosine similarity has no defined direction for it. How might a production embedding pipeline handle empty inputs?
5. If the optional pretrained model is available, compare its top five with the teaching encoder's. Which results are better for your information need, and what evidence supports that judgment?
6. Propose a query set with at least five queries and relevance labels to compare lexical and dense methods. Include one paraphrase and one exact identifier query.
7. Explain why a high cosine score should not be read as “this chunk has an X% chance of being correct.”

### Optional extension: validate the vector calculations

These checks cover the properties expected from the small implementation: equal dimensions, normalized nonzero vectors, deterministic ranking, and explicit handling of a negative K.

In [13]:
assert all(len(vector) == len(DIMENSION_NAMES) for vector in chunk_vectors.values())
assert all(abs(math.sqrt(dot_product(v, v)) - 1.0) < 1e-12
           for v in chunk_vectors.values() if any(v))
assert dense_rank(query, corpus, chunk_vectors) == dense_rank(query, corpus, chunk_vectors)
try:
    dense_retrieve(query, corpus, chunk_vectors, k=-1)
except ValueError:
    print('Negative K is rejected as expected.')
else:
    raise AssertionError('Negative K should have raised ValueError')

print('Vector dimension, normalization, deterministic ranking, and K checks passed.')

Negative K is rejected as expected.
Vector dimension, normalization, deterministic ranking, and K checks passed.


## Summary

Dense retrieval encodes corpus chunks and queries into vectors in a compatible space, compares those vectors, ranks the candidates, and returns top-K. Cosine similarity measures vector direction; normalized vectors allow dot-product scoring to produce the same cosine values. A real encoder learns representations from data, while the tiny concept encoder in this lesson only demonstrates the mechanics and is not a production model.

Dense retrieval can help bridge different wording, but it can still miss exact identifiers, rank topical non-answers, and fail when the model or chunking is a poor fit. Scores are not probabilities. Inspect results and use relevance judgments and metrics to compare methods. The next notebook introduces **sparse retrieval and BM25**, which bring stronger lexical matching and can complement dense search in hybrid retrieval.